# 🎙️ BroDub
Foreign voices ➜ English, in the same character's voice. Runs on Google's free GPU.

In [ ]:
#@title ▶ BroDub: press the play button
#@markdown Pick a video and it comes back in English, with the characters' own voices. About 20-30 min for one 24 min episode.
use_google_drive = False #@param {type:"boolean"}
#@markdown ☝️ For big videos or a lot of episodes: tick this, put the videos in Google Drive in a folder called **BroDub**, and the English ones show up in **BroDub/English**.

import os, subprocess, sys, threading
from pathlib import Path

ENGINE = '#!/usr/bin/env python3\n"""BroDub: turns the foreign voices in a video into English, in each character\'s own voice.\n\n1. Split the voices from the music + sounds (Demucs), so the AI only hears people talking.\n2. Hear every line and when it\'s said (faster-whisper, with word timings).\n3. Translate the lines into casual spoken English that fits the time (Qwen3 on a GPU).\n4. Find out who\'s talking: lines with the same voice become one character.\n5. Say each line in English with that character\'s voice (Chatterbox voice cloning).\n6. Fit each English line into its time slot, cut the original voice there, and mix it over the\n   untouched music. Laughs, screams and songs keep the original voice.\n\n    python brodub.py episode.mp4                # -> "episode (English dub).mp4" next to it\n    python brodub.py a.mp4 b.mkv --out-dir done/\n"""\nimport argparse\nimport gc\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nSR = 44100  # mixing rate\nASR_SR = 16000\nCHARS_PER_SEC = 14.0  # how fast the English voice talks, roughly (used to keep lines short)\nSONG = re.compile(r"[♪♫♬]")\n# Language-free sounds: the original voice stays for these (real dubs keep the grunts too).\nGRUNT = re.compile(r"^(?:(?:a+h*|o+h*|u+h*|uh+|um+|hm+|mm+|huh|eh+|ugh+|ha+|ha(?:ha)+|heh|hehe|whoa|wow|ooh|aah|hah|tch|ow+|ah-?ha)[\\s,.!?~…-]*)+$", re.I)\n# Everyday anime phrases a small translator gets too literal.\nJA_TIPS = (\n    "しょうがない/仕方ない = oh well / fine, whatever; 言い訳 = (making) excuses; よろしく = nice to meet you / count on "\n    "you; お疲れ = good work; なるほど = I see; まさか = no way; うそ = no way / you\'re kidding; やばい = oh crap / "\n    "awesome; すごい = amazing; ちょっと待って = hold on; 大丈夫 = you okay? / I\'m fine; バカ = idiot; うるさい = shut up; "\n    "いただきます = let\'s eat; ただいま/おかえり = I\'m home / welcome back; 行ってきます = I\'m heading out; 頑張って = "\n    "you got this; ごめん = sorry; ありがとう = thanks; 何で = why; 何だと = what did you say; くそ = damn; "\n    "せんぱい/先輩 = senpai; ～さん/くん/ちゃん/様 = keep as -san/-kun/-chan/-sama"\n)\nLANG_NAMES = {\n    "ja": "Japanese", "ko": "Korean", "zh": "Chinese", "es": "Spanish", "pt": "Portuguese", "fr": "French",\n    "de": "German", "it": "Italian", "ru": "Russian", "th": "Thai", "vi": "Vietnamese", "id": "Indonesian",\n    "tl": "Tagalog", "hi": "Hindi", "ar": "Arabic", "tr": "Turkish", "pl": "Polish", "nl": "Dutch",\n}\n\n\ndef say(msg):\n    print(msg, flush=True)\n\n\ndef free_gpu():\n    gc.collect()\n    try:\n        import torch\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    except ImportError:\n        pass\n\n\ndef ffmpeg(*args):\n    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", *args], check=True)\n\n\ndef load_audio(path, sr, mono):\n    """Any audio/video file -> float32 samples (frames x channels, or 1-D when mono)."""\n    ch = 1 if mono else 2\n    raw = subprocess.run(\n        ["ffmpeg", "-hide_banner", "-loglevel", "error", "-i", str(path), "-vn", "-ac", str(ch),\n         "-ar", str(sr), "-f", "f32le", "-"],\n        check=True, capture_output=True,\n    ).stdout\n    a = np.frombuffer(raw, dtype=np.float32).copy()\n    return a if mono else a.reshape(-1, 2)\n\n\ndef write_wav(path, audio, sr):\n    import soundfile as sf\n    sf.write(str(path), audio, sr, subtype="FLOAT")\n\n\ndef rms_db(x):\n    if len(x) == 0:\n        return -120.0\n    return float(20 * np.log10(np.sqrt(np.mean(np.square(x, dtype=np.float64))) + 1e-9))\n\n\ndef has_rubberband():\n    out = subprocess.run(["ffmpeg", "-hide_banner", "-filters"], capture_output=True, text=True).stdout\n    return " rubberband " in out\n\n\n# ---------------------------------------------------------------- 1. voices away from the music\n\ndef split_voices(src_wav, work, device):\n    model = "htdemucs_ft" if device == "cuda" else "htdemucs"\n    from demucs.separate import main as demucs_main\n    demucs_main(["--two-stems", "vocals", "-n", model, "-d", device, "--float32", "-o", str(work / "sep"), str(src_wav)])\n    out = work / "sep" / model / src_wav.stem\n    return out / "vocals.wav", out / "no_vocals.wav"\n\n\n# ---------------------------------------------------------------- 2. hear the lines\n\ndef hear_lines(vocals16, device, model_name, lang):\n    from faster_whisper import WhisperModel\n    model = WhisperModel(model_name, device=device, compute_type="float16" if device == "cuda" else "int8")\n    segments, info = model.transcribe(\n        vocals16, language=lang, task="transcribe", beam_size=5, word_timestamps=True,\n        vad_filter=True, vad_parameters={"min_silence_duration_ms": 350},\n        condition_on_previous_text=False, language_detection_segments=8,\n    )\n    lines = []\n    for seg in segments:\n        unsure = seg.no_speech_prob > 0.6 and seg.avg_logprob < -1.0\n        cur = []\n        for w in seg.words or []:\n            ends = cur[-1].word.strip()[-1:] if cur else ""\n            if cur and (w.start - cur[-1].end > 0.55 or (ends and ends in "。！？!?." and cur[-1].end - cur[0].start > 1.0)\n                        or w.end - cur[0].start > 12):\n                lines.append(_line(cur, unsure, seg.text))\n                cur = []\n            cur.append(w)\n        if cur:\n            lines.append(_line(cur, unsure, seg.text))\n    lines = [ln for ln in lines if ln["text"] and ln["end"] - ln["start"] >= 0.25]\n    del model\n    free_gpu()\n    return lines, info.language\n\n\ndef measure_lines(lines, vocals16, music16):\n    for ln in lines:\n        a, b = int(ln["start"] * ASR_SR), int(ln["end"] * ASR_SR)\n        ln["voice_db"] = rms_db(vocals16[a:b])\n        ln["clean"] = ln["voice_db"] - rms_db(music16[a:b])  # how far the voice is above the music\n\n\ndef mark_songs(lines):\n    """Opening/ending songs: the singing is about as loud as the music (talking sits well above it) and it\n    goes on and on. Those keep the original singer."""\n    loud = [ln for ln in lines if ln["clean"] < 4]\n    for ln in loud:\n        mid = (ln["start"] + ln["end"]) / 2\n        near = [o for o in loud if abs((o["start"] + o["end"]) / 2 - mid) < 15]\n        sung = sum(min(o["end"], mid + 15) - max(o["start"], mid - 15) for o in near)\n        if len(near) >= 4 and sung > 30 * 0.6:\n            ln["song"] = True\n\n\ndef _line(words, unsure, seg_text):\n    text = "".join(w.word for w in words).strip()\n    return {\n        "start": float(words[0].start), "end": float(words[-1].end), "text": text,\n        "song": bool(SONG.search(seg_text)) or not re.search(r"\\w", text), "unsure": unsure,\n    }\n\n\n# ---------------------------------------------------------------- 3. translate\n\nclass Translator:\n    def __init__(self, model_name, device, src_lang):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n        self.torch = torch\n        self.tok = AutoTokenizer.from_pretrained(model_name)\n        dtype = torch.float16 if device == "cuda" else torch.bfloat16\n        self.model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype).to(device).eval()\n        self.device = device\n        self.lang = LANG_NAMES.get(src_lang, src_lang)\n        self.src = src_lang\n        self.cast = ""\n\n    def find_names(self, texts):\n        """Character names in the whole episode, written in English, so every line spells them the same."""\n        text = "\\n".join(texts)[:12000]\n        system = (\n            f"These lines come from a speech recognizer listening to a {self.lang} show (punctuation can be missing). "\n            "List the names of the people/characters that appear, written the usual English way (like Kenta, Yui). "\n            "Reply with only the names separated by commas, or \'none\'."\n        )\n        names = self._ask(system, text, 60)\n        self.cast = "" if names.lower().startswith("none") else names\n\n    def _ask(self, system, user, max_new):\n        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]\n        prompt = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        ids = self.tok(prompt, return_tensors="pt").to(self.device)\n        with self.torch.inference_mode():\n            out = self.model.generate(**ids, max_new_tokens=max_new, do_sample=False, repetition_penalty=1.05)\n        text = self.tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)\n        return clean_reply(text)\n\n    def line(self, before, text, after, secs):\n        context = "".join(f"{b[\'text\']}  =>  {b.get(\'english\', \'\')}\\n" for b in before)\n        nxt = f"\\nNext lines (don\'t translate them): {after}" if after else ""\n        system = (\n            f"You translate {self.lang} anime dialogue into English for a dub. Write what an English voice actor "\n            "would say: natural, casual spoken English with the same feeling (yelling stays yelling). Keep names "\n            "as they are. Match the length of the original so it fits the mouth movements. Reply with only the "\n            "English line, no quotes, no notes. The lines come from a speech recognizer, so punctuation can be "\n            "missing: work out who is being talked to from the meaning. Translate the meaning, not word by word."\n            + (f" Names in this show: {self.cast}." if self.cast else "")\n            + (f" Phrases: {JA_TIPS}." if self.src == "ja" else "")\n        )\n        user = f"Earlier lines:\\n{context or \'(none)\'}{nxt}\\n\\nTranslate this line ({secs:.1f} seconds long):\\n{text}"\n        return self._ask(system, user, 80)\n\n    def shorter(self, english, words):\n        system = "You shorten English dub lines. Keep the meaning and the feeling. Reply with only the new line."\n        return self._ask(system, f"Say this in at most {words} words:\\n{english}", 60)\n\n    def close(self):\n        del self.model\n        free_gpu()\n\n\ndef clean_reply(text):\n    text = text.strip().split("\\n")[0].strip()\n    text = re.sub(r"^(english|translation|dub)\\s*:\\s*", "", text, flags=re.I)\n    return text.strip().strip(\'"“”「」『』\').strip()\n\n\ndef translate_lines(lines, src, device, llm, whisper_name, vocals16):\n    todo = [ln for ln in lines if not ln["song"]]\n    tr = None\n    if llm != "none":\n        try:\n            tr = Translator(llm, device, src)\n        except Exception as e:  # out of memory, no download, ...: Whisper\'s translator does it all\n            say(f"   (the translator AI didn\'t load: {e.__class__.__name__}, using Whisper\'s instead)")\n    if tr is not None:\n        t0 = time.time()\n        tr.find_names([ln["text"] for ln in todo])\n        if tr.cast:\n            say(f"   names: {tr.cast}")\n        for i, ln in enumerate(todo):\n            after = " / ".join(o["text"] for o in todo[i + 1:i + 3])\n            ln["english"] = tr.line(todo[max(0, i - 4):i], ln["text"], after, ln["end"] - ln["start"])\n            room = room_for(ln, lines)\n            if len(ln["english"]) / CHARS_PER_SEC > room * 1.4:\n                short = tr.shorter(ln["english"], max(2, int(room * CHARS_PER_SEC / 5.5)))\n                if re.search(r"[A-Za-z]", short) and len(short) < len(ln["english"]):\n                    ln["english"] = short\n            if i % 20 == 0:\n                say(f"   {i + 1}/{len(todo)}  {ln[\'text\']}  ->  {ln[\'english\']}")\n        tr.close()\n        say(f"   translated {len(todo)} lines in {time.time() - t0:.0f} s")\n    # Whisper\'s own translator, straight from the voice: for everything when there\'s no translator AI,\n    # else only for lines where it gave back junk.\n    left = [ln for ln in todo if not re.search(r"[A-Za-z]", ln.get("english", ""))]\n    if not left:\n        return\n    from faster_whisper import WhisperModel\n    model = WhisperModel(whisper_name, device=device, compute_type="float16" if device == "cuda" else "int8")\n    for ln in left:\n        a, b = int(ln["start"] * ASR_SR), int(ln["end"] * ASR_SR)\n        segs, _ = model.transcribe(vocals16[a:b], language=src, task="translate", beam_size=5)\n        ln["english"] = " ".join(s.text.strip() for s in segs).strip()\n    del model\n    free_gpu()\n\n\ndef room_for(ln, lines):\n    """Seconds the English may use: until the next line starts, but not much longer than the original."""\n    nxt = [o["start"] for o in lines if o["start"] > ln["start"] + 0.05]\n    until_next = (min(nxt) - ln["start"] - 0.08) if nxt else 1e9\n    dur = ln["end"] - ln["start"]\n    return max(dur, min(until_next, max(dur * 1.25, dur + 0.4)))\n\n\n# ---------------------------------------------------------------- 4. who\'s talking\n\ndef find_characters(lines, vocals16, ve):\n    from sklearn.cluster import AgglomerativeClustering\n    idx = [i for i, ln in enumerate(lines) if ln.get("dub") and ln["end"] - ln["start"] >= 0.6]\n    clips = [vocals16[int(lines[i]["start"] * ASR_SR):int(lines[i]["end"] * ASR_SR)] for i in idx]\n    emb = np.zeros((0, 256), np.float32)\n    if clips:\n        emb = ve.embeds_from_wavs(clips, sample_rate=ASR_SR)\n        emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)\n    if len(idx) >= 2:\n        labels = AgglomerativeClustering(n_clusters=None, metric="cosine", linkage="average",\n                                         distance_threshold=0.32).fit(emb).labels_\n    else:\n        labels = np.zeros(len(idx), int)\n    for i, lab in zip(idx, labels):\n        lines[i]["who"] = int(lab)\n    names = sorted(set(int(x) for x in labels))\n    cents = {k: emb[labels == k].mean(0) for k in names}\n    # Short lines: the closest-sounding character (if it can be heard at all), else the last speaker.\n    last = names[0] if names else 0\n    for ln in lines:\n        if not ln.get("dub"):\n            continue\n        if "who" not in ln:\n            clip = vocals16[int(ln["start"] * ASR_SR):int(ln["end"] * ASR_SR)]\n            guess = None\n            if cents and len(clip) > ASR_SR * 0.25:\n                try:\n                    e = ve.embeds_from_wavs([clip], sample_rate=ASR_SR)[0]\n                    guess = max(cents, key=lambda k: float(np.dot(cents[k], e)))\n                except Exception:\n                    guess = None\n            ln["who"] = guess if guess is not None else last\n        last = ln["who"]\n    return sorted(set(ln["who"] for ln in lines if ln.get("dub")))\n\n\ndef voice_sample(lines, who, vocals16, path):\n    """About 10 s of this character\'s clearest talking (the AI copies the voice from it)."""\n    mine = [ln for ln in lines if ln.get("dub") and ln["who"] == who]\n    good = sorted(mine, key=lambda ln: (ln["end"] - ln["start"] >= 1.0, ln["clean"]), reverse=True)\n    parts, total = [], 0.0\n    gap = np.zeros(int(ASR_SR * 0.15), np.float32)\n    for ln in good:\n        a, b = int(ln["start"] * ASR_SR), int((ln["end"] + 0.05) * ASR_SR)\n        clip = vocals16[a:b]\n        parts += [clip, gap]\n        total += len(clip) / ASR_SR\n        if total >= 10:\n            break\n    audio = np.concatenate(parts) if parts else np.zeros(ASR_SR, np.float32)\n    audio = audio * (10 ** ((-20 - rms_db(audio)) / 20))\n    write_wav(path, np.clip(audio, -1, 1), ASR_SR)\n\n\n# ---------------------------------------------------------------- 5. English voices\n\ndef speak_lines(lines, chars, vocals16, work, tts):\n    import torch\n    by_who = {}\n    for ln in lines:\n        if ln.get("dub"):\n            by_who.setdefault(ln["who"], []).append(ln)\n    done, total, t0 = 0, sum(len(v) for v in by_who.values()), time.time()\n    for who in chars:\n        ref = work / f"voice_{who}.wav"\n        voice_sample(lines, who, vocals16, ref)\n        tts.prepare_conditionals(str(ref), exaggeration=0.5)\n        mine = by_who.get(who, [])\n        loud = np.median([ln["voice_db"] for ln in mine]) if mine else -20\n        for ln in mine:\n            # Louder than this character usually talks = more feeling (yelling), quieter = calmer.\n            ex = float(np.clip(0.5 + (ln["voice_db"] - loud) / 14, 0.3, 0.95))\n            want = len(ln["english"]) / CHARS_PER_SEC\n            best = None\n            for attempt in range(3):\n                torch.manual_seed(1234 + done * 7 + attempt)\n                wav = tts.generate(ln["english"], exaggeration=ex, cfg_weight=0.5 if ex < 0.7 else 0.35,\n                                   temperature=0.75 - 0.1 * attempt)\n                y = wav.squeeze(0).cpu().numpy().astype(np.float32)\n                secs = len(y) / tts.sr\n                ok = 0.25 < secs < want * 2.2 + 1.5  # way too long = the AI rambled\n                if best is None or abs(secs - want) < abs(best[1] - want):\n                    best = (y, secs)\n                if ok:\n                    break\n            ln["_voice"] = best[0]\n            done += 1\n            if done % 10 == 0 or done == total:\n                left = (time.time() - t0) / done * (total - done)\n                say(f"   {done}/{total} lines spoken  (about {left / 60:.0f} min left)")\n\n\n# ---------------------------------------------------------------- 6. fit + mix\n\ndef fit_line(y, tts_sr, room, use_rb):\n    import librosa\n    y, _ = librosa.effects.trim(y, top_db=40)\n    y = librosa.resample(y, orig_sr=tts_sr, target_sr=SR)\n    secs = len(y) / SR\n    if secs > room > 0:\n        rate = min(secs / room, 1.35)\n        if rate > 1.02:\n            y = stretch(y, rate, use_rb)\n    fade = min(len(y) // 4, int(SR * 0.012))\n    if fade > 0:\n        ramp = np.linspace(0, 1, fade, dtype=np.float32)\n        y[:fade] *= ramp\n        y[-fade:] *= ramp[::-1]\n    return y\n\n\ndef stretch(y, rate, use_rb):\n    with tempfile.TemporaryDirectory() as d:\n        a, b = Path(d) / "a.wav", Path(d) / "b.wav"\n        write_wav(a, y, SR)\n        filt = f"rubberband=tempo={rate:.4f}:formant=preserved" if use_rb else f"atempo={rate:.4f}"\n        ffmpeg("-i", str(a), "-af", filt, "-ar", str(SR), "-ac", "1", str(b))\n        return load_audio(b, SR, mono=True)\n\n\ndef mix(lines, vocals, music, use_rb, tts_sr):\n    n = max(len(vocals), len(music))\n    vocals = np.pad(vocals, ((0, n - len(vocals)), (0, 0)))\n    music = np.pad(music, ((0, n - len(music)), (0, 0)))\n    keep = np.ones(n, np.float32)\n    english = np.zeros(n, np.float32)\n    ramp = int(SR * 0.03)\n    for ln in lines:\n        if "_voice" not in ln:\n            continue\n        y = fit_line(ln.pop("_voice"), tts_sr, room_for(ln, lines), use_rb)\n        a = int(ln["start"] * SR)\n        orig = vocals[a:int(ln["end"] * SR)].mean(1)\n        gain = 10 ** (np.clip(rms_db(orig) - rms_db(y), -12, 12) / 20)\n        b = min(n, a + len(y))\n        english[a:b] += y[:b - a] * gain\n        ln["english_end"] = b / SR\n        # Cut the original voice under the line (a bit wider: word timings end early).\n        ca, cb = max(0, int((ln["start"] - 0.06) * SR)), min(n, int(max(ln["end"] + 0.12, b / SR) * SR))\n        keep[ca:cb] = 0\n    # Smooth the cuts so they don\'t click.\n    edges = np.flatnonzero(np.diff(keep) != 0)\n    smooth = keep.copy()\n    for e in edges:\n        lo, hi = max(0, e - ramp), min(n, e + ramp)\n        smooth[lo:hi] = np.minimum(smooth[lo:hi], np.linspace(keep[lo], keep[hi - 1], hi - lo))\n    out = music + vocals * smooth[:, None] + english[:, None]\n    # Soft limiter: no clipping when the English is louder than the original.\n    over = np.abs(out) > 0.8\n    out[over] = np.sign(out[over]) * (0.8 + 0.15 * np.tanh((np.abs(out[over]) - 0.8) / 0.15))\n    return out.astype(np.float32)\n\n\ndef srt_time(t):\n    ms = int(round(t * 1000))\n    return f"{ms // 3600000:02d}:{ms // 60000 % 60:02d}:{ms // 1000 % 60:02d},{ms % 1000:03d}"\n\n\ndef write_srt(lines, path):\n    rows = [ln for ln in lines if ln.get("english")]\n    with open(path, "w", encoding="utf-8") as f:\n        for i, ln in enumerate(rows, 1):\n            end = max(ln["end"], ln.get("english_end", ln["end"]))\n            f.write(f"{i}\\n{srt_time(ln[\'start\'])} --> {srt_time(end)}\\n{ln[\'english\']}\\n\\n")\n\n\ndef put_together(src, mix_wav, srt, out):\n    subs = srt.stat().st_size > 0\n    common = ["-map", "1:a:0", "-map", "0:a:0?", *(["-map", "2:s:0", "-c:s", "mov_text"] if subs else []),\n              "-c:a", "aac", "-b:a", "192k",\n              "-metadata:s:a:0", "language=eng", "-metadata:s:a:0", "title=English dub",\n              "-metadata:s:a:1", "title=Original", "-disposition:a:0", "default", "-disposition:a:1", "0",\n              *(["-metadata:s:s:0", "language=eng", "-disposition:s:0", "0"] if subs else []),\n              "-movflags", "+faststart", str(out)]\n    inputs = ["-i", str(src), "-i", str(mix_wav), *(["-i", str(srt)] if subs else [])]\n    try:\n        ffmpeg(*inputs, "-map", "0:v:0", "-c:v", "copy", *common)\n    except subprocess.CalledProcessError:  # a video kind mp4 can\'t hold as-is: re-make the picture\n        ffmpeg(*inputs, "-map", "0:v:0", "-c:v", "libx264", "-crf", "18", "-preset", "fast", "-pix_fmt", "yuv420p", *common)\n\n\n# ---------------------------------------------------------------- main\n\ndef dub(src, out, args, tts_cache):\n    t_start = time.time()\n    device = args.device\n    work = Path(tempfile.mkdtemp(prefix="brodub-", dir=args.work))\n    say(f"\\n🎬 {src.name}")\n    orig = work / "orig.wav"\n    cut = ["-t", str(args.max_seconds)] if args.max_seconds else []\n    ffmpeg("-i", str(src), *cut, "-vn", "-ac", "2", "-ar", str(SR), "-c:a", "pcm_f32le", str(orig))\n\n    say("1/6 Taking the voices out of the music...")\n    voc_path, mus_path = split_voices(orig, work, device)\n    free_gpu()\n    vocals, music = load_audio(voc_path, SR, mono=False), load_audio(mus_path, SR, mono=False)\n    vocals16, music16 = load_audio(voc_path, ASR_SR, mono=True), load_audio(mus_path, ASR_SR, mono=True)\n\n    say("2/6 Listening to every line...")\n    lines, lang = hear_lines(vocals16, device, args.whisper, args.lang)\n    measure_lines(lines, vocals16, music16)\n    mark_songs(lines)\n    songs = sum(ln["song"] for ln in lines)\n    say(f"   heard {len(lines)} lines in {LANG_NAMES.get(lang, lang)}" + (f" ({songs} sung, those stay as they are)" if songs else ""))\n    if lang == "en":\n        say("   it\'s already English: nothing to dub, skipping this one")\n        shutil.rmtree(work, ignore_errors=True)\n        return None\n\n    say("3/6 Translating...")\n    translate_lines(lines, lang, device, args.llm, args.whisper, vocals16)\n    for ln in lines:\n        e = ln.get("english", "")\n        ln["dub"] = bool(e) and not ln["song"] and not ln["unsure"] and not GRUNT.match(e) and bool(re.search(r"[A-Za-z]", e))\n\n    say("4/6 Finding who\'s talking...")\n    if "tts" not in tts_cache:\n        from chatterbox.tts import ChatterboxTTS\n        tts_cache["tts"] = ChatterboxTTS.from_pretrained(device=device)\n    tts = tts_cache["tts"]\n    chars = find_characters(lines, vocals16, tts.ve)\n    say(f"   {len(chars)} different voices")\n\n    say("5/6 Saying it in English with their voices...")\n    speak_lines(lines, chars, vocals16, work, tts)\n\n    say("6/6 Putting it back together...")\n    mixed = mix(lines, vocals, music, has_rubberband(), tts.sr)\n    mix_wav, srt = work / "mix.wav", work / "english.srt"\n    write_wav(mix_wav, mixed, SR)\n    write_srt(lines, srt)\n    put_together(src, mix_wav, srt, out)\n    shutil.copy(srt, out.with_suffix(".srt"))\n    with open(work / "lines.json", "w", encoding="utf-8") as f:\n        json.dump(lines, f, ensure_ascii=False, indent=1)\n    dubbed = sum(1 for ln in lines if ln.get("dub"))\n    say(f"✅ {out.name}: {dubbed} lines in English, {len(chars)} voices, took {(time.time() - t_start) / 60:.1f} min")\n    if not args.keep_work:\n        shutil.rmtree(work, ignore_errors=True)\n    return out\n\n\ndef main():\n    p = argparse.ArgumentParser(description="BroDub: dub a video\'s voices into English, keeping each character\'s voice.")\n    p.add_argument("videos", nargs="+", type=Path)\n    p.add_argument("--out-dir", type=Path, help="where the English videos go (default: next to each video)")\n    p.add_argument("--lang", default=None, help="language spoken in the video, like ja (default: figure it out)")\n    p.add_argument("--whisper", default=None, help="speech model (default: large-v3 on a GPU, small without)")\n    p.add_argument("--llm", default="Qwen/Qwen3-4B-Instruct-2507", help="translator model, or \'none\' for Whisper\'s")\n    p.add_argument("--device", default=None, help="cuda or cpu (default: cuda if there is one)")\n    p.add_argument("--work", default=None, help="folder for the in-between files")\n    p.add_argument("--keep-work", action="store_true", help="keep the in-between files (for fixing bugs)")\n    p.add_argument("--max-seconds", type=float, default=0, help="only do the first N seconds (quick test)")\n    args = p.parse_args()\n\n    import torch\n    if args.device is None:\n        args.device = "cuda" if torch.cuda.is_available() else "cpu"\n    if args.whisper is None:\n        args.whisper = "large-v3" if args.device == "cuda" else "small"\n    if args.device == "cpu":\n        say("⚠️ No GPU: this works, but it\'s slow (hours for one episode).")\n    tts_cache, made = {}, []\n    for src in args.videos:\n        if not src.exists():\n            say(f"❌ can\'t find {src}")\n            continue\n        folder = args.out_dir or src.parent\n        folder.mkdir(parents=True, exist_ok=True)\n        out = folder / f"{src.stem} (English dub).mp4"\n        try:\n            r = dub(src, out, args, tts_cache)\n            if r:\n                made.append(r)\n        except subprocess.CalledProcessError as e:\n            say(f"❌ {src.name}: couldn\'t read or write the video ({e.cmd[0]} failed). Is it a real video file?")\n    if not made:\n        sys.exit(1)\n\n\nif __name__ == "__main__":\n    main()\n'
VIDEO = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v", ".ts", ".flv", ".3gp"}
ENV = "/content/bd"

if "GPU" not in subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout:
    raise SystemExit("😵 Google didn't give a free GPU this time (they run out sometimes). Wait a bit and press ▶ again.")

Path("/content/brodub.py").write_text(ENGINE, encoding="utf-8")

def setup():
    if Path(ENV, "ready").exists():
        return
    cmds = [
        "pip -q install uv",
        f"uv venv -q -p 3.12 {ENV}",
        f"uv pip install -q -p {ENV}/bin/python torch==2.6.0 torchaudio==2.6.0 chatterbox-tts==0.1.7 "
        "faster-whisper==1.2.1 ctranslate2==4.5.0 demucs==4.1.0",
    ]
    for c in cmds:
        r = subprocess.run(c, shell=True, capture_output=True, text=True)
        if r.returncode:
            setup.error = (r.stderr or r.stdout)[-1500:]
            return
    Path(ENV, "ready").touch()
setup.error = None

worker = threading.Thread(target=setup)
worker.start()

if use_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    folder = Path("/content/drive/MyDrive/BroDub")
    out_dir = folder / "English"
    out_dir.mkdir(parents=True, exist_ok=True)
    videos = sorted(p for p in folder.iterdir() if p.suffix.lower() in VIDEO
                    and not (out_dir / f"{p.stem} (English dub).mp4").exists())
    if not videos:
        raise SystemExit("📂 No new videos in Google Drive > BroDub. Put some there and press ▶ again.")
else:
    from google.colab import files
    Path("/content/in").mkdir(exist_ok=True)
    os.chdir("/content/in")
    print("👇 Pick your video")
    videos = [Path("/content/in", n) for n in files.upload()]
    out_dir = Path("/content/out")
    if not videos:
        raise SystemExit("No video picked. Press ▶ again.")

if not Path(ENV, "ready").exists():
    print("⏳ Getting the AI ready (only the first time, about 3 min)...")
worker.join()
if setup.error:
    raise SystemExit("😵 Setting up failed:\n" + setup.error)

env = dict(os.environ)
nv = Path(ENV, "lib/python3.12/site-packages/nvidia")
env["LD_LIBRARY_PATH"] = f"{nv}/cudnn/lib:{nv}/cublas/lib:" + env.get("LD_LIBRARY_PATH", "")
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
env["TQDM_DISABLE"] = "1"
NOISE = ("Selected model", "Separated tracks", "Separating track", "Warning")
errlog = open("/content/brodub-log.txt", "w")
run = subprocess.Popen([f"{ENV}/bin/python", "/content/brodub.py", *map(str, videos), "--out-dir", str(out_dir)],
                       stdout=subprocess.PIPE, stderr=errlog, text=True, env=env)
for line in run.stdout:
    if line.strip() and not line.startswith(NOISE):
        print(line, end="")
if run.wait():
    errlog.close()
    print(open("/content/brodub-log.txt").read()[-3000:])
    raise SystemExit("😵 Something broke. Send Claude a screenshot of this box.")

made = [out_dir / f"{v.stem} (English dub).mp4" for v in videos]
made = [m for m in made if m.exists()]
if use_google_drive:
    print(f"🍿 Done! Your English videos are in Google Drive > BroDub > English ({len(made)} new)")
else:
    print("🍿 Done! Downloading it now...")
    for m in made:
        files.download(str(m))
